<a href="https://colab.research.google.com/github/Shreya08-cyber/CSA6101-digital-forensics/blob/main/network_traffic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**

To analyze simulated firewall/network connection logs and detect reconnaissance activity (port scanning) based on a single IP targeting multiple distinct ports in a short timeframe.

**Algorithm**

Read network traffic entries containing source IP, destination IP, and target port.

Set a detection threshold for unique destination ports probed by a single source (e.g., > 3 ports).

Create a dictionary to map each source IP to a set of target ports.

Iterate through each network traffic record.

Parse the source IP address and target port from the entry.

Add the target port to the respective source IP's set to filter duplicate hits to the same port.

Check the size of each source IP's set against the defined port threshold.

Display an alert for any source IP probing a suspicious number of distinct ports.

In [1]:

traffic_logs = [
    {"src_ip": "172.16.0.45", "dest_ip": "10.0.0.1", "dest_port": 22},
    {"src_ip": "172.16.0.45", "dest_ip": "10.0.0.1", "dest_port": 80},
    {"src_ip": "172.16.0.45", "dest_ip": "10.0.0.1", "dest_port": 443},
    {"src_ip": "172.16.0.45", "dest_ip": "10.0.0.1", "dest_port": 8080},
    {"src_ip": "10.0.0.5",     "dest_ip": "10.0.0.1", "dest_port": 80},
    {"src_ip": "10.0.0.5",     "dest_ip": "10.0.0.1", "dest_port": 80}, # Duplicate connection
]

PORT_THRESHOLD = 3
ip_port_tracker = {}

for log in traffic_logs:
    ip = log["src_ip"]
    port = log["dest_port"]

    if ip not in ip_port_tracker:
        ip_port_tracker[ip] = set()
    ip_port_tracker[ip].add(port)

print("--- PORT SCAN DETECTION RESULTS ---")
for ip, ports in ip_port_tracker.items():
    unique_port_count = len(ports)
    if unique_port_count > PORT_THRESHOLD:
        print(f"[ALERT] Port scanning detected from IP: {ip}")
        print(f"        Targeted {unique_port_count} distinct ports: {sorted(list(ports))}")
    else:
        print(f"[OK] Normal traffic from IP: {ip} (Probed {unique_port_count} port)")

--- PORT SCAN DETECTION RESULTS ---
[ALERT] Port scanning detected from IP: 172.16.0.45
        Targeted 4 distinct ports: [22, 80, 443, 8080]
[OK] Normal traffic from IP: 10.0.0.5 (Probed 1 port)


**Result**

The script groups connection attempts by source IP address and flags 172.16.0.45 for contacting 4 unique destination ports (22, 80, 443, 8080), correctly identifying port-scanning behavior while letting legitimate web traffic (10.0.0.5) pass.